# 🏋️ Решения задач по Pandas

Этот ноутбук содержит решения практических задач из туториала.  
Каждая задача разбита на условие и решение с пояснениями.

**Датасет:** пассажиры Титаника + таблица портов посадки.

In [ ]:
import pandas as pd
import numpy as np

# Загрузка данных (тот же код, что и в туториале)
try:
    url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
    titanic = pd.read_csv(url)
    print("Загружено из интернета")
except Exception:
    rng = np.random.default_rng(42)
    n = 600
    titanic = pd.DataFrame({
        'PassengerId': range(1, n + 1),
        'Survived': rng.integers(0, 2, n),
        'Pclass': rng.integers(1, 4, n),
        'Name': [f'Passenger {i}' for i in range(n)],
        'Sex': rng.choice(['male', 'female'], n),
        'Age': np.round(rng.normal(30, 14, n), 1),
        'SibSp': rng.integers(0, 6, n),
        'Parch': rng.integers(0, 3, n),
        'Fare': np.round(np.abs(rng.normal(50, 40, n)), 2),
        'Embarked': rng.choice(['S', 'C', 'Q'], n),
        'Cabin': np.nan,
    })
    print("Сгенерирован локальный датасет")

# Подготовка: восстановим оригинальные имена столбцов, если они были переименованы
for old, new in [('siblings_spouses', 'SibSp'), ('parents_children', 'Parch')]:
    if new not in titanic.columns and old in titanic.columns:
        titanic = titanic.rename(columns={old: new})

# Заполнение пропусков
if 'Age' in titanic.columns:
    titanic['Age'] = titanic['Age'].fillna(titanic['Age'].median())
if 'Embarked' in titanic.columns:
    titanic['Embarked'] = titanic['Embarked'].fillna(titanic['Embarked'].mode()[0])
if 'Cabin' in titanic.columns:
    pass  # оставим как есть для задачи 9

# Таблица портов (из туториала)
ports = pd.DataFrame({
    'code': ['S', 'C', 'Q'],
    'port_name': ['Саутгемптон', 'Шербур', 'Квинстаун'],
    'country': ['UK', 'France', 'Ireland']
})

print(f"Строк: {titanic.shape[0]}, столбцов: {titanic.shape[1]}")
titanic.head(2)

---

## Задача 1: Самый молодой и самый старший пассажир

**Условие:** Найдите самого молодого и самого старшего пассажира.  
**Подсказка:** используйте `idxmin()` и `idxmax()`, затем `.loc[]`.

In [ ]:
# idxmin / idxmax возвращают ИНДЕКС строки с минимальным/максимальным значением
youngest_idx = titanic['Age'].idxmin()
oldest_idx = titanic['Age'].idxmax()

print("Самый молодой:")
print(titanic.loc[youngest_idx, ['Name', 'Age', 'Sex', 'Pclass']])

print("
Самый старший:")
print(titanic.loc[oldest_idx, ['Name', 'Age', 'Sex', 'Pclass']])

# Альтернатива: sort_values (медленнее, но нагляднее)
# titanic.nsmallest(1, 'Age')[['Name', 'Age']]
# titanic.nlargest(1, 'Age')[['Name', 'Age']]

---

## Задача 2: Средний возраст выживших и погибших

**Условие:** Посчитайте средний возраст для пассажиров, у которых `Survived` = 1 (выжили) и `Survived` = 0 (погибли).

In [ ]:
# Группировка по Survived и агрегация среднего возраста
age_by_survival = titanic.groupby('Survived')['Age'].mean().round(2)
print(age_by_survival)

# Более красивый вывод с подписями
print(f"
Средний возраст погибших (0): {age_by_survival[0]} лет")
print(f"Средний возраст выживших (1): {age_by_survival[1]} лет")

# Альтернатива: pivot_table
# titanic.pivot_table(values='Age', index='Survived', aggfunc='mean')

---

## Задача 3: Порт с самой высокой выживаемостью

**Условие:** Определите, какой порт посадки (`Embarked`) давал самую высокую долю выживших.

In [ ]:
# Группировка по порту и среднее Survived (для 0/1 среднее = доля выживших)
survival_by_port = titanic.groupby('Embarked')['Survived'].mean().sort_values(ascending=False)
print(survival_by_port.round(3))

best_port = survival_by_port.idxmax()
print(f"
Лучший порт: {best_port} (выживаемость {survival_by_port[best_port]:.2%})")

# Сопоставление с названием порта
port_names = {'S': 'Саутгемптон', 'C': 'Шербур', 'Q': 'Квинстаун'}
print(f"Это порт: {port_names.get(best_port, best_port)}")

---

## Задача 4: Размер семьи и выживаемость

**Условие:** Создайте столбец `FamilySize = SibSp + Parch + 1`.  
Сравните выживаемость для одиночек (`FamilySize == 1`) и семей (`FamilySize > 1`).

In [ ]:
# Создание столбца FamilySize
titanic['FamilySize'] = titanic['SibSp'] + titanic['Parch'] + 1

# Фильтрация и подсчёт выживаемости
alone = titanic[titanic['FamilySize'] == 1]
family = titanic[titanic['FamilySize'] > 1]

print(f"Одиночек: {len(alone)}, выживаемость: {alone['Survived'].mean():.2%}")
print(f"Семей: {len(family)}, выживаемость: {family['Survived'].mean():.2%}")

# Через groupby (более элегантно)
titanic['is_alone'] = np.where(titanic['FamilySize'] == 1, 'одиночка', 'семья')
print("
Через groupby:")
print(titanic.groupby('is_alone')['Survived'].mean().round(3))

---

## Задача 5: Join с таблицей портов — пассажиры из Шербура

**Условие:** Сделайте join таблиц с именами портов. Выведите всех людей, кто сел в порту «Шербур».

In [ ]:
# Join (merge) по ключу Embarked = code
titanic_ports = titanic.merge(ports, left_on='Embarked', right_on='code', how='left')

# Фильтрация по названию порта
cherbourg = titanic_ports[titanic_ports['port_name'] == 'Шербур']
print(f"Пассажиров из Шербура: {len(cherbourg)}")

# Выводим нужные колонки
display(cherbourg[['Name', 'Age', 'Sex', 'Pclass', 'Embarked', 'port_name', 'country']].head(10))

# Альтернатива через query
# titanic_ports.query("port_name == 'Шербур'")[[...]]

---

## Задача 6: Возрастные группы и выживаемость (cut)

**Условие:** Используя `pd.cut`, разбейте пассажиров на 4 возрастные группы и посчитайте выживаемость в каждой.

In [ ]:
# pd.cut создаёт равные интервалы (или указанные вручную)
titanic['age_group'] = pd.cut(
    titanic['Age'],
    bins=[0, 18, 35, 60, 100],
    labels=['ребёнок (0-18)', 'молодой (18-35)', 'взрослый (35-60)', 'пожилой (60+)']
)

# Выживаемость по группам
survival_by_age = titanic.groupby('age_group', observed=True)['Survived'].agg(['mean', 'count'])
survival_by_age['mean'] = survival_by_age['mean'].round(3)
survival_by_age.columns = ['выживаемость', 'количество']
print(survival_by_age)

# observed=True — для категориальных данных (не показывать пустые категории)

---

## Задача 7: Сводная таблица Sex × Pclass

**Условие:** С помощью `pivot_table` постройте таблицу: строки — `Sex`, столбцы — `Pclass`, значения — средняя `Fare`.

In [ ]:
# pivot_table: строки = Sex, столбцы = Pclass, значения = среднее Fare
fare_table = titanic.pivot_table(
    values='Fare',
    index='Sex',
    columns='Pclass',
    aggfunc='mean'
).round(2)

print("Средняя цена билета (Fare) по полу и классу:")
display(fare_table)

# С итогами
fare_table_with_margins = titanic.pivot_table(
    values='Fare',
    index='Sex',
    columns='Pclass',
    aggfunc='mean',
    margins=True
).round(2)
print("
С итогами:")
display(fare_table_with_margins)

---

## Задача 8: Аномалии — дорогие билеты в третьем классе

**Условие:** Найдите пассажиров третьего класса, возрастом от 20 до 40 лет, заплативших больше среднего Fare людей из второго класса.

In [ ]:
# Шаг 1: средняя цена билета во 2-м классе
mean_fare_2nd = titanic[titanic['Pclass'] == 2]['Fare'].mean()
print(f"Средний Fare во 2-м классе: {mean_fare_2nd:.2f}")

# Шаг 2: фильтр аномалий
anomalies = titanic[
    (titanic['Pclass'] == 3) &
    (titanic['Age'].between(20, 40)) &
    (titanic['Fare'] > mean_fare_2nd)
]

print(f"
Найдено аномалий: {len(anomalies)}")
display(anomalies[['Name', 'Age', 'Pclass', 'Fare']].head(10))

# Сравним средние
print(f"
Средний Fare аномалий: {anomalies['Fare'].mean():.2f}")
print(f"Средний Fare обычных 3-го класса: {titanic[titanic['Pclass'] == 3]['Fare'].mean():.2f}")

---

## Задача 9: Палуба из номера каюты и выживаемость

**Условие:** Создайте столбец `deck` из первой буквы `Cabin` (если `Cabin` есть в данных). Посчитайте выживаемость по палубам.  
*Примечание:* в реальном датасете Титаника `Cabin` много пропусков.

In [ ]:
# Проверяем, есть ли Cabin и какие там значения
print("Пропусков в Cabin:", titanic['Cabin'].isna().sum())
print("Примеры непустых значений:")
print(titanic['Cabin'].dropna().head())

# Извлекаем первую букву (палубу)
# na=False в str[0] не работает, поэтому используем .astype(str).str[0]
# но для NaN лучше через .str accessor с fillna
titanic['deck'] = titanic['Cabin'].astype(str).str[0]

# NaN превращается в 'n' (от 'nan'), заменим обратно на NaN
titanic.loc[titanic['Cabin'].isna(), 'deck'] = np.nan

print("
Распределение по палубам:")
print(titanic['deck'].value_counts(dropna=False))

# Выживаемость по палубам (только для тех, у кого указана каюта)
deck_survival = titanic.dropna(subset=['deck']).groupby('deck')['Survived'].agg(['mean', 'count']).round(3)
deck_survival.columns = ['выживаемость', 'количество']
print("
Выживаемость по палубам:")
print(deck_survival)

---

## Задача 10: Разбивка по диапазонам возрастов (value_counts)

**Условие:** Сделайте разбивку людей по диапазонам возрастов `[0, 18, 35, 60, 100]` и посчитайте `value_counts` (сколько человек в каждой группе).

In [ ]:
# pd.cut создаёт категории, value_counts считает частоту
titanic['age_range'] = pd.cut(
    titanic['Age'],
    bins=[0, 18, 35, 60, 100],
    labels=['0-18', '18-35', '35-60', '60+'],
    right=False   # интервалы [0, 18) — полуоткрытые слева
)

# value_counts с сортировкой по индексу (а не по частоте)
age_counts = titanic['age_range'].value_counts().sort_index()
print("Количество пассажиров по возрастным группам:")
print(age_counts)

# Доля от общего числа
print("
Доли:")
print((age_counts / age_counts.sum()).round(3))

# Красивый бар-график
age_counts.plot.bar(title='Распределение пассажиров по возрастам', rot=0, color='teal')
import matplotlib.pyplot as plt
plt.ylabel('Количество')
plt.show()

---

## ✅ Все задачи решены!

**Ключевые приёмы, которые здесь отработаны:**
- `idxmin` / `idxmax` + `loc` для поиска экстремумов
- `groupby` + `mean` / `agg` для агрегаций
- `merge` (JOIN) для объединения таблиц
- `pd.cut` для биннинга непрерывных переменных
- `pivot_table` для сводных таблиц
- Сложная фильтрация через `&`, `between`, `>`
- `str` accessor для работы с текстом (`str[0]`)
- `value_counts` для частотного анализа
- `np.where` для создания бинарных флагов

Не стесняйтесь менять параметры, добавлять визуализации и экспериментировать! 🚀